# Road PCI Intelligence — Step 2: Manifests, Mask Conversion & Leak-Proof Splits

This notebook establishes the foundational data ingestion pipeline for the **Road PCI Intelligence** project:
1. **Manifest Construction**: Builds unified and per-source CSV manifests in `manifests/` with relative paths.
2. **Standardized Mask Conversion Rules**:
   - **`road-crack-dataset`**: Value 1 $\to$ Crack; Values 2 & 3 $\to$ Background (Construction joints and core drilling).
   - **`CRACK500`**: Any non-zero $\to$ Crack.
   - **`pothole`**: YOLOv8 polygon vector vertices rasterized to binary mask via `cv2.fillPoly`.
   - **`RDD2022ES`**: Bounding boxes only, reserved for downstream validation / detection.
3. **Leak-Proof Group Partitioning**:
   - **Pothole**: Base group ID `pic-<N>` (augmented variations strictly co-located). Val split = 15% of train groups; test split = official validation directory.
   - **Road-Crack**: Group ID = scene (`road_<N>` / `Production_1_ortho_merge`). Scene-level 70/15/15 partition.
   - **CRACK500**: Preserves official benchmark split lists (`train.lst`, `val.lst`, `test.lst`).
   - **RDD2022ES**: Strips `xmirror_` prefix so horizontally flipped pairs strictly share the same split (80/10/10).
4. **Verification Test Suite**: Runs unit tests validating zero group leakage, 100% path disk existence, split counts, and mirrored pair integrity.
5. **Foreground Pixel Stats & Visual Verification**: Computes defect pixel fractions per source and split, and visualizes 4 samples per segmentation source.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & REPO STRUCTURE INITIALIZATION
# ==============================================================================
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Create required directories
Path('src').mkdir(parents=True, exist_ok=True)
Path('tests').mkdir(parents=True, exist_ok=True)
Path('manifests').mkdir(parents=True, exist_ok=True)

# Ensure package inits exist
Path('src/__init__.py').write_text('# src package\n', encoding='utf-8')
Path('tests/__init__.py').write_text('# tests package\n', encoding='utf-8')

# Ensure modules exist on disk (for self-contained Kaggle execution)
data_rules_path = Path('src/data_rules.py')
if not data_rules_path.exists() or data_rules_path.stat().st_size == 0:
    data_rules_path.write_text('"""\nData Conversion Rules, Assumptions, and Label Processing for Road PCI Intelligence.\n\nThis module defines deterministic conversion rules from raw dataset annotations\ninto standardized segmentation masks (crack and pothole channels) and bounding\nbox representations.\n"""\n\nfrom pathlib import Path\nfrom typing import Tuple, List, Optional\nimport numpy as np\nimport cv2\nfrom PIL import Image\n\n# Channel identification constants\nCHANNEL_CRACK = 0\nCHANNEL_POTHOLE = 1\nNUM_SEGMENTATION_CHANNELS = 2\n\n# Source-specific annotation capabilities\nSOURCE_CAPABILITIES = {\n    \'road_crack\': {\n        \'labels_crack\': True,\n        \'labels_pothole\': False,\n        \'annotation_type\': \'raster_mask\'\n    },\n    \'crack500\': {\n        \'labels_crack\': True,\n        \'labels_pothole\': False,\n        \'annotation_type\': \'raster_mask\'\n    },\n    \'pothole\': {\n        \'labels_crack\': False,\n        \'labels_pothole\': True,\n        \'annotation_type\': \'yolo_polygon\'\n    },\n    \'rdd2022es\': {\n        \'labels_crack\': False,\n        \'labels_pothole\': False,\n        \'annotation_type\': \'yolo_bbox\'\n    }\n}\n\n\ndef convert_road_crack_mask(raw_mask: np.ndarray) -> np.ndarray:\n    """\n    Converts raw road-crack-dataset multi-class mask to a binary crack mask (0 or 1).\n\n    RULE & ASSUMPTION:\n    - Value 1 -> 1 (Crack).\n    - Value 2 -> 0 (Background).\n    - Value 3 -> 0 (Background).\n    - Value 0 -> 0 (Background).\n\n    ASSUMPTION DOCUMENTATION:\n    The dataset author\'s Kaggle metadata lists classes in order:\n    1. Concrete Crack\n    2. Construction Joints\n    3. Drilling\n    However, the on-disk repository contains an empty README.md and no config file.\n    Empirical inspection reveals:\n    - Value 1 features elongated, tortuous crack networks (72.4% touch image borders).\n    - Value 2 features straight, artificial construction slab joint seams (94.8% touch borders).\n    - Value 3 features isolated, small circular core-drilling holes (0.0% touch borders).\n    Therefore, Value 1 is mapped to \'crack\', while construction joints (2) and drill holes (3)\n    are mapped to background. This is a recorded assumption:\n    "value-to-class mapping inferred from the Kaggle description order and mask geometry,\n    not documented in the files".\n\n    Args:\n        raw_mask: 2D numpy array with integer pixel values in {0, 1, 2, 3}.\n\n    Returns:\n        binary_mask: 2D uint8 numpy array with values in {0, 1}.\n    """\n    binary_mask = np.zeros(raw_mask.shape[:2], dtype=np.uint8)\n    binary_mask[raw_mask == 1] = 1\n    return binary_mask\n\n\ndef convert_crack500_mask(raw_mask: np.ndarray) -> np.ndarray:\n    """\n    Converts raw CRACK500 ground-truth mask to a binary crack mask (0 or 1).\n\n    RULE:\n    - Any non-zero pixel (> 0) -> 1 (Crack).\n    - Zero (0) -> 0 (Background).\n\n    CRACK500 benchmark stores masks as binary grayscale PNGs (0 and 255).\n    Any non-zero defect pixel is converted to standard class index 1.\n\n    Args:\n        raw_mask: 2D numpy array.\n\n    Returns:\n        binary_mask: 2D uint8 numpy array with values in {0, 1}.\n    """\n    binary_mask = np.zeros(raw_mask.shape[:2], dtype=np.uint8)\n    binary_mask[raw_mask > 0] = 1\n    return binary_mask\n\n\ndef rasterize_pothole_polygons(\n    img_shape: Tuple[int, int],\n    label_path: Path\n) -> np.ndarray:\n    """\n    Rasterizes YOLOv8 normalized polygon annotations into a binary pothole mask (0 or 1).\n\n    RULE:\n    - Reads line-by-line normalized coordinates (class_id x1 y1 x2 y2 ... xn yn).\n    - Denormalizes polygon vertices to image coordinates (width, height).\n    - Fills polygons using cv2.fillPoly to generate binary mask (1 for pothole, 0 for background).\n\n    Args:\n        img_shape: (height, width) of the target image.\n        label_path: Path to the YOLOv8 .txt polygon annotation file.\n\n    Returns:\n        binary_mask: 2D uint8 numpy array with values in {0, 1}.\n    """\n    h, w = img_shape[:2]\n    binary_mask = np.zeros((h, w), dtype=np.uint8)\n\n    if not label_path.exists() or label_path.stat().st_size == 0:\n        return binary_mask\n\n    with open(label_path, \'r\', encoding=\'utf-8\') as f:\n        lines = f.readlines()\n\n    for line in lines:\n        parts = line.strip().split()\n        if len(parts) > 4:  # At least class_id + 2 coordinate pairs\n            coords = [float(x) for x in parts[1:]]\n            pts = []\n            for i in range(0, len(coords), 2):\n                px = int(round(coords[i] * w))\n                py = int(round(coords[i + 1] * h))\n                pts.append([px, py])\n            if len(pts) >= 3:\n                pts_arr = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))\n                cv2.fillPoly(binary_mask, [pts_arr], color=1)\n\n    return binary_mask\n\n\ndef load_standardized_mask(\n    source: str,\n    mask_or_label_path: Path,\n    img_shape: Optional[Tuple[int, int]] = None\n) -> np.ndarray:\n    """\n    Loads and applies the corresponding source conversion rule to return a\n    standardized binary mask (values in {0, 1}).\n\n    Args:\n        source: Name of the source (\'road_crack\', \'crack500\', \'pothole\').\n        mask_or_label_path: Absolute or relative Path to mask image or label text file.\n        img_shape: (height, width) tuple required for polygon rasterization.\n\n    Returns:\n        2D uint8 mask with values in {0, 1}.\n    """\n    if source == \'road_crack\':\n        raw = np.array(Image.open(mask_or_label_path))\n        return convert_road_crack_mask(raw)\n\n    elif source == \'crack500\':\n        raw = np.array(Image.open(mask_or_label_path))\n        return convert_crack500_mask(raw)\n\n    elif source == \'pothole\':\n        if img_shape is None:\n            raise ValueError("img_shape (height, width) is required for rasterizing pothole polygons.")\n        return rasterize_pothole_polygons(img_shape, mask_or_label_path)\n\n    else:\n        raise ValueError(f"Unknown segmentation source: {source}")\n', encoding='utf-8')
    print('[OK] Written src/data_rules.py')

build_manifest_path = Path('src/build_manifest.py')
if not build_manifest_path.exists() or build_manifest_path.stat().st_size == 0:
    build_manifest_path.write_text('"""\nManifest Builder for Road PCI Intelligence.\n\nGenerates leak-proof, group-aware dataset manifests for road distress segmentation\nand object detection tasks. All paths in manifests are recorded relative to the\nrespective dataset root to maintain portability across Kaggle and local environments.\n"""\n\nimport os\nimport sys\nimport re\nimport random\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nfrom typing import Dict, List, Tuple, Optional\n\n# Ensure project root is in sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nimport pandas as pd\nfrom PIL import Image\n\ntry:\n    from src.data_rules import SOURCE_CAPABILITIES\nexcept ImportError:\n    from data_rules import SOURCE_CAPABILITIES\n\n# Global random seed for reproducible splits\nRANDOM_SEED = 42\n\n# Dataset registry definitions\nDATASET_SPECS = {\n    \'pothole\': {\n        \'owner\': \'farzadnekouei\',\n        \'name\': \'pothole-image-segmentation-dataset\',\n        \'display_name\': \'Pothole Image Segmentation\'\n    },\n    \'road_crack\': {\n        \'owner\': \'rukiyeaydn\',\n        \'name\': \'road-crack-dataset\',\n        \'display_name\': \'Road Crack Dataset\'\n    },\n    \'crack500\': {\n        \'owner\': \'sj26717\',\n        \'name\': \'crack500\',\n        \'display_name\': \'CRACK500\'\n    },\n    \'rdd2022es\': {\n        \'owner\': \'juusos\',\n        \'name\': \'rdd2022es\',\n        \'display_name\': \'RDD2022ES\'\n    }\n}\n\n\ndef resolve_dataset_root(owner: str, name: str) -> Optional[Path]:\n    """\n    Locates the dataset root directory on Kaggle or locally.\n    Checks /kaggle/input/datasets/<owner>/<name>/ first, then fallback paths.\n    """\n    candidates = [\n        Path(\'/kaggle/input/datasets\') / owner / name,\n        Path(\'/kaggle/input\') / owner / name,\n        Path(\'/kaggle/input\') / name,\n        Path(\'data\') / name,\n        Path(\'../data\') / name\n    ]\n    for c in candidates:\n        if c.exists() and c.is_dir():\n            return c.resolve()\n\n    # Recursive fallback\n    k_input = Path(\'/kaggle/input\')\n    if k_input.exists():\n        for d in k_input.rglob(name):\n            if d.is_dir():\n                return d.resolve()\n\n    return None\n\n\ndef get_image_dimensions(img_path: Path) -> Tuple[int, int]:\n    """Reads image (width, height) without loading the full raster pixel buffer."""\n    try:\n        with Image.open(img_path) as img:\n            return img.size  # (width, height)\n    except Exception:\n        return (640, 640)\n\n\n# ==============================================================================\n# 1. POTHOLE DATASET MANIFEST BUILDER\n# ==============================================================================\ndef build_pothole_manifest(dataset_root: Path, seed: int = RANDOM_SEED) -> pd.DataFrame:\n    """\n    Builds manifest for farzadnekouei/pothole-image-segmentation-dataset.\n\n    Split rules:\n    - group_id = \'pic-<N>\' from filename (e.g. \'pic-282\').\n    - test = official valid/ folder (un-augmented validation scenes).\n    - val = 15% of train group_ids (all augmented variants stay together).\n    - train = remaining 85% of train group_ids.\n    - labels_crack = False, labels_pothole = True.\n    """\n    records = []\n    train_dir = dataset_root / \'Pothole_Segmentation_YOLOv8\' / \'train\'\n    valid_dir = dataset_root / \'Pothole_Segmentation_YOLOv8\' / \'valid\'\n\n    # Fallback to search if directory structure varies slightly\n    if not train_dir.exists():\n        matches = list(dataset_root.rglob(\'train\'))\n        if matches:\n            train_dir = matches[0]\n            valid_dir = train_dir.parent / \'valid\'\n\n    train_imgs = sorted(list((train_dir / \'images\').glob(\'*.jpg\')))\n    valid_imgs = sorted(list((valid_dir / \'images\').glob(\'*.jpg\')))\n\n    def get_pothole_group_id(filename: str) -> str:\n        m = re.match(r\'^(pic-\\d+)\', filename, re.IGNORECASE)\n        return m.group(1).lower() if m else Path(filename).stem\n\n    # 1. Assign test split from official valid/ directory\n    for img_p in valid_imgs:\n        label_p = (valid_dir / \'labels\' / f"{img_p.stem}.txt")\n        rel_img = img_p.relative_to(dataset_root).as_posix()\n        rel_lbl = label_p.relative_to(dataset_root).as_posix() if label_p.exists() else ""\n        w, h = get_image_dimensions(img_p)\n        gid = get_pothole_group_id(img_p.name)\n\n        records.append({\n            \'source\': \'pothole\',\n            \'image_path\': rel_img,\n            \'mask_path_or_label_path\': rel_lbl,\n            \'group_id\': gid,\n            \'split\': \'test\',\n            \'width\': w,\n            \'height\': h,\n            \'labels_crack\': False,\n            \'labels_pothole\': True\n        })\n\n    # 2. Partition train/ directory into train and val by group_id\n    train_group_to_imgs = defaultdict(list)\n    for img_p in train_imgs:\n        gid = get_pothole_group_id(img_p.name)\n        train_group_to_imgs[gid].append(img_p)\n\n    unique_train_groups = sorted(list(train_group_to_imgs.keys()))\n    rng = random.Random(seed)\n    shuffled_groups = unique_train_groups.copy()\n    rng.shuffle(shuffled_groups)\n\n    n_val = max(1, int(round(len(shuffled_groups) * 0.15)))\n    val_groups = set(shuffled_groups[:n_val])\n    train_groups = set(shuffled_groups[n_val:])\n\n    for gid, img_list in train_group_to_imgs.items():\n        split = \'val\' if gid in val_groups else \'train\'\n        for img_p in img_list:\n            label_p = (train_dir / \'labels\' / f"{img_p.stem}.txt")\n            rel_img = img_p.relative_to(dataset_root).as_posix()\n            rel_lbl = label_p.relative_to(dataset_root).as_posix() if label_p.exists() else ""\n            w, h = get_image_dimensions(img_p)\n\n            records.append({\n                \'source\': \'pothole\',\n                \'image_path\': rel_img,\n                \'mask_path_or_label_path\': rel_lbl,\n                \'group_id\': gid,\n                \'split\': split,\n                \'width\': w,\n                \'height\': h,\n                \'labels_crack\': False,\n                \'labels_pothole\': True\n            })\n\n    df = pd.DataFrame(records)\n    return df\n\n\n# ==============================================================================\n# 2. ROAD-CRACK DATASET MANIFEST BUILDER\n# ==============================================================================\ndef build_road_crack_manifest(dataset_root: Path, seed: int = RANDOM_SEED) -> pd.DataFrame:\n    """\n    Builds manifest for rukiyeaydn/road-crack-dataset.\n\n    Split rules:\n    - group_id = scene (e.g. \'road_<N>\' or \'Production_1_ortho_merge\').\n    - Partition scenes targeting approximately 70% train / 15% val / 15% test by image count.\n    - All crops/tiles of a scene go to the exact same split (zero scene leakage).\n    - labels_crack = True, labels_pothole = False.\n    """\n    records = []\n    # Collect all image files (excluding _mask.png)\n    img_files = []\n    for root, _, files in os.walk(dataset_root):\n        for f in files:\n            p = Path(root) / f\n            if p.suffix.lower() in {\'.jpg\', \'.png\'} and not f.endswith(\'_mask.png\') and \'_mask\' not in p.stem:\n                img_files.append(p)\n\n    def extract_scene_group_id(filename: str) -> str:\n        stem = Path(filename).stem\n        if \'.rf.\' in filename:\n            key1 = filename.split(\'.rf.\')[0]\n        else:\n            key1 = stem\n        key1 = re.sub(r\'(_jpg|_png)$\', \'\', key1, flags=re.IGNORECASE)\n\n        m_road = re.match(r\'^(road_\\d+)\', key1)\n        if m_road:\n            return m_road.group(1)\n        elif key1.startswith(\'Production_1_ortho_merge\'):\n            return \'Production_1_ortho_merge\'\n        else:\n            return re.sub(r\'(_\\d+)+$\', \'\', key1)\n\n    scene_to_images = defaultdict(list)\n    for img_p in img_files:\n        scene = extract_scene_group_id(img_p.name)\n        scene_to_images[scene].append(img_p)\n\n    total_images = len(img_files)\n    target_train = total_images * 0.70\n    target_val = total_images * 0.15\n    target_test = total_images * 0.15\n\n    # Deterministic scene allocation\n    rng = random.Random(seed)\n    scene_items = sorted(list(scene_to_images.items()), key=lambda x: x[0])\n    rng.shuffle(scene_items)\n\n    scene_split_map = {}\n    train_c, val_c, test_c = 0, 0, 0\n\n    for sc, imgs in scene_items:\n        count = len(imgs)\n        # Allocate to the split furthest below its target percentage\n        ratios = [\n            (train_c / target_train, 0),\n            (val_c / target_val, 1),\n            (test_c / target_test, 2)\n        ]\n        ratios.sort(key=lambda x: x[0])\n        chosen_split = ratios[0][1]\n\n        if chosen_split == 0:\n            scene_split_map[sc] = \'train\'\n            train_c += count\n        elif chosen_split == 1:\n            scene_split_map[sc] = \'val\'\n            val_c += count\n        else:\n            scene_split_map[sc] = \'test\'\n            test_c += count\n\n    print(f"\\n[road-crack] Scene Allocation (Total: {total_images} images across {len(scene_to_images)} scenes):")\n    print(f"  • Train: {train_c} images ({train_c/total_images*100:.1f}%) -> {sorted([sc for sc, sp in scene_split_map.items() if sp==\'train\'])}")\n    print(f"  • Val:   {val_c} images ({val_c/total_images*100:.1f}%) -> {sorted([sc for sc, sp in scene_split_map.items() if sp==\'val\'])}")\n    print(f"  • Test:  {test_c} images ({test_c/total_images*100:.1f}%) -> {sorted([sc for sc, sp in scene_split_map.items() if sp==\'test\'])}")\n\n    for sc, imgs in scene_to_images.items():\n        split = scene_split_map[sc]\n        for img_p in imgs:\n            # Mask matching: image.jpg -> image_mask.png\n            mask_name = f"{img_p.stem}_mask.png"\n            mask_p = img_p.parent / mask_name\n            if not mask_p.exists():\n                # Alternative search\n                base_name = img_p.name.replace(\'.jpg\', \'\').replace(\'.png\', \'\')\n                cands = list(img_p.parent.glob(f"{base_name}*mask.png"))\n                mask_p = cands[0] if cands else None\n\n            rel_img = img_p.relative_to(dataset_root).as_posix()\n            rel_mask = mask_p.relative_to(dataset_root).as_posix() if mask_p and mask_p.exists() else ""\n            w, h = get_image_dimensions(img_p)\n\n            records.append({\n                \'source\': \'road_crack\',\n                \'image_path\': rel_img,\n                \'mask_path_or_label_path\': rel_mask,\n                \'group_id\': sc,\n                \'split\': split,\n                \'width\': w,\n                \'height\': h,\n                \'labels_crack\': True,\n                \'labels_pothole\': False\n            })\n\n    df = pd.DataFrame(records)\n    return df\n\n\n# ==============================================================================\n# 3. CRACK500 DATASET MANIFEST BUILDER\n# ==============================================================================\ndef build_crack500_manifest(dataset_root: Path) -> pd.DataFrame:\n    """\n    Builds manifest for sj26717/crack500.\n\n    Split rules:\n    - Uses official CRACK500/splits/train.lst, val.lst, test.lst.\n    - group_id = base image stem (e.g. \'20160222_165402\').\n    - labels_crack = True, labels_pothole = False.\n    """\n    records = []\n    splits_dir = dataset_root / \'CRACK500\' / \'splits\'\n    if not splits_dir.exists():\n        splits_dirs = list(dataset_root.rglob(\'splits\'))\n        splits_dir = splits_dirs[0] if splits_dirs else dataset_root\n\n    split_files = {\n        \'train\': splits_dir / \'train.lst\',\n        \'val\': splits_dir / \'val.lst\',\n        \'test\': splits_dir / \'test.lst\'\n    }\n\n    for split_name, lst_path in split_files.items():\n        if not lst_path.exists():\n            print(f"Warning: {lst_path} does not exist!")\n            continue\n\n        with open(lst_path, \'r\', encoding=\'utf-8\') as f:\n            lines = [line.strip() for line in f if line.strip()]\n\n        for line in lines:\n            parts = line.split()\n            if len(parts) >= 2:\n                rel_img_raw = parts[0]\n                rel_mask_raw = parts[1]\n            else:\n                rel_img_raw = parts[0]\n                rel_mask_raw = parts[0].replace(\'JPEGImages/images/\', \'Annotations/masks/\').replace(\'.jpg\', \'_mask.png\')\n\n            # Build full paths under dataset_root\n            img_p = dataset_root / \'CRACK500\' / rel_img_raw\n            mask_p = dataset_root / \'CRACK500\' / rel_mask_raw\n            if not img_p.exists():\n                img_p = dataset_root / rel_img_raw\n                mask_p = dataset_root / rel_mask_raw\n\n            w, h = get_image_dimensions(img_p) if img_p.exists() else (2000, 1500)\n            gid = Path(rel_img_raw).stem\n\n            # Record relative to dataset_root\n            rel_img = img_p.relative_to(dataset_root).as_posix() if img_p.exists() else rel_img_raw\n            rel_mask = mask_p.relative_to(dataset_root).as_posix() if mask_p.exists() else rel_mask_raw\n\n            records.append({\n                \'source\': \'crack500\',\n                \'image_path\': rel_img,\n                \'mask_path_or_label_path\': rel_mask,\n                \'group_id\': gid,\n                \'split\': split_name,\n                \'width\': w,\n                \'height\': h,\n                \'labels_crack\': True,\n                \'labels_pothole\': False\n            })\n\n    df = pd.DataFrame(records)\n    return df\n\n\n# ==============================================================================\n# 4. RDD2022ES DATASET MANIFEST BUILDER (BOUNDING BOXES ONLY)\n# ==============================================================================\ndef build_rdd2022es_manifest(dataset_root: Path, seed: int = RANDOM_SEED) -> pd.DataFrame:\n    """\n    Builds manifest for juusos/rdd2022es (Bounding box dataset).\n\n    Split rules:\n    - group_id = filename without \'xmirror_\' prefix (e.g. \'Czech_000000\').\n    - 80% train / 10% val / 10% test partitioned deterministically by group_id.\n    - Mirrored pairs (\'xmirror_...\') strictly share the exact same split as their original.\n    - labels_crack = False, labels_pothole = False (not used for segmentation loss).\n    """\n    records = []\n    annotated_dir = dataset_root / \'combined_annotatedv2\'\n    if not annotated_dir.exists():\n        dirs = list(dataset_root.rglob(\'combined_annotatedv2\'))\n        annotated_dir = dirs[0] if dirs else dataset_root\n\n    img_files = sorted(list(annotated_dir.glob(\'*.jpg\')))\n\n    def get_rdd_group_id(filename: str) -> str:\n        stem = Path(filename).stem\n        if stem.startswith(\'xmirror_\'):\n            return stem[len(\'xmirror_\'):]\n        return stem\n\n    # Group images by original scene\n    group_to_imgs = defaultdict(list)\n    for img_p in img_files:\n        gid = get_rdd_group_id(img_p.name)\n        group_to_imgs[gid].append(img_p)\n\n    unique_groups = sorted(list(group_to_imgs.keys()))\n    rng = random.Random(seed)\n    shuffled_groups = unique_groups.copy()\n    rng.shuffle(shuffled_groups)\n\n    n_total = len(shuffled_groups)\n    n_train = int(round(n_total * 0.80))\n    n_val = int(round(n_total * 0.10))\n\n    train_groups = set(shuffled_groups[:n_train])\n    val_groups = set(shuffled_groups[n_train:n_train + n_val])\n    test_groups = set(shuffled_groups[n_train + n_val:])\n\n    for gid, imgs in group_to_imgs.items():\n        if gid in train_groups:\n            split = \'train\'\n        elif gid in val_groups:\n            split = \'val\'\n        else:\n            split = \'test\'\n\n        for img_p in imgs:\n            txt_p = img_p.with_suffix(\'.txt\')\n            rel_img = img_p.relative_to(dataset_root).as_posix()\n            rel_txt = txt_p.relative_to(dataset_root).as_posix() if txt_p.exists() else ""\n            w, h = get_image_dimensions(img_p)\n\n            records.append({\n                \'source\': \'rdd2022es\',\n                \'image_path\': rel_img,\n                \'mask_path_or_label_path\': rel_txt,\n                \'group_id\': gid,\n                \'split\': split,\n                \'width\': w,\n                \'height\': h,\n                \'labels_crack\': False,\n                \'labels_pothole\': False\n            })\n\n    df = pd.DataFrame(records)\n    return df\n\n\n# ==============================================================================\n# MAIN DRIVER: GENERATE AND EXPORT ALL MANIFESTS\n# ==============================================================================\ndef generate_all_manifests(output_dir: Path = Path(\'manifests\')) -> Dict[str, pd.DataFrame]:\n    """\n    Discovers all 4 datasets, generates their manifests, and exports CSV files\n    to output_dir (root/manifests/).\n    """\n    output_dir.mkdir(parents=True, exist_ok=True)\n    manifests = {}\n\n    print("=" * 80)\n    print("ROAD PCI INTELLIGENCE: BUILDING DATASET MANIFESTS")\n    print(f"Output Directory: {output_dir.resolve()}")\n    print("=" * 80)\n\n    # 1. Pothole\n    spec_p = DATASET_SPECS[\'pothole\']\n    root_p = resolve_dataset_root(spec_p[\'owner\'], spec_p[\'name\'])\n    if root_p:\n        print(f"\\nBuilding manifest for Pothole dataset ({root_p})...")\n        df_p = build_pothole_manifest(root_p)\n        df_p.to_csv(output_dir / \'pothole_manifest.csv\', index=False)\n        manifests[\'pothole\'] = df_p\n        print(f"Saved {output_dir / \'pothole_manifest.csv\'} ({len(df_p)} records)")\n    else:\n        print(f"Warning: Root for {spec_p[\'name\']} not found!")\n\n    # 2. Road Crack\n    spec_rc = DATASET_SPECS[\'road_crack\']\n    root_rc = resolve_dataset_root(spec_rc[\'owner\'], spec_rc[\'name\'])\n    if root_rc:\n        print(f"\\nBuilding manifest for Road-Crack dataset ({root_rc})...")\n        df_rc = build_road_crack_manifest(root_rc)\n        df_rc.to_csv(output_dir / \'road_crack_manifest.csv\', index=False)\n        manifests[\'road_crack\'] = df_rc\n        print(f"Saved {output_dir / \'road_crack_manifest.csv\'} ({len(df_rc)} records)")\n    else:\n        print(f"Warning: Root for {spec_rc[\'name\']} not found!")\n\n    # 3. CRACK500\n    spec_c = DATASET_SPECS[\'crack500\']\n    root_c = resolve_dataset_root(spec_c[\'owner\'], spec_c[\'name\'])\n    if root_c:\n        print(f"\\nBuilding manifest for CRACK500 dataset ({root_c})...")\n        df_c = build_crack500_manifest(root_c)\n        df_c.to_csv(output_dir / \'crack500_manifest.csv\', index=False)\n        manifests[\'crack500\'] = df_c\n        print(f"Saved {output_dir / \'crack500_manifest.csv\'} ({len(df_c)} records)")\n    else:\n        print(f"Warning: Root for {spec_c[\'name\']} not found!")\n\n    # 4. Unified Segmentation Manifest\n    seg_dfs = [df for k, df in manifests.items() if k in {\'pothole\', \'road_crack\', \'crack500\'}]\n    if seg_dfs:\n        df_seg = pd.concat(seg_dfs, ignore_index=True)\n        df_seg.to_csv(output_dir / \'segmentation_manifest.csv\', index=False)\n        manifests[\'segmentation\'] = df_seg\n        print(f"\\nSaved Unified {output_dir / \'segmentation_manifest.csv\'} ({len(df_seg)} records)")\n\n    # 5. RDD2022ES (Bounding Box manifest)\n    spec_r = DATASET_SPECS[\'rdd2022es\']\n    root_r = resolve_dataset_root(spec_r[\'owner\'], spec_r[\'name\'])\n    if root_r:\n        print(f"\\nBuilding manifest for RDD2022ES dataset ({root_r})...")\n        df_r = build_rdd2022es_manifest(root_r)\n        df_r.to_csv(output_dir / \'rdd2022es_manifest.csv\', index=False)\n        manifests[\'rdd2022es\'] = df_r\n        print(f"Saved {output_dir / \'rdd2022es_manifest.csv\'} ({len(df_r)} records)")\n    else:\n        print(f"Warning: Root for {spec_r[\'name\']} not found!")\n\n    # Print Summary Table\n    print("\\n" + "=" * 80)\n    print("MANIFEST SUMMARY TABLE")\n    print("=" * 80)\n    summary_rows = []\n    for src_key, df in manifests.items():\n        if src_key == \'segmentation\':\n            continue\n        for sp in [\'train\', \'val\', \'test\']:\n            sub = df[df[\'split\'] == sp]\n            summary_rows.append({\n                \'Source\': src_key,\n                \'Split\': sp,\n                \'Images\': len(sub),\n                \'Unique Groups\': sub[\'group_id\'].nunique(),\n                \'Crack Channel\': sub[\'labels_crack\'].iloc[0] if len(sub) > 0 else False,\n                \'Pothole Channel\': sub[\'labels_pothole\'].iloc[0] if len(sub) > 0 else False\n            })\n    summary_df = pd.DataFrame(summary_rows)\n    print(summary_df.to_string(index=False))\n\n    return manifests\n\n\nif __name__ == \'__main__\':\n    generate_all_manifests()\n', encoding='utf-8')
    print('[OK] Written src/build_manifest.py')

test_manifest_path = Path('tests/test_manifest.py')
if not test_manifest_path.exists() or test_manifest_path.stat().st_size == 0:
    test_manifest_path.write_text('"""\nTest Suite for Road PCI Intelligence Dataset Manifests.\n\nAssertions:\n(a) No group_id appears in more than one split within any source (Zero data leakage).\n(b) Every file path recorded in the manifests actually exists on disk.\n(c) Split sizes match the expected partition counts.\n(d) Mirrored RDD2022ES pairs strictly share the exact same split.\n"""\n\nimport sys\nimport unittest\nfrom pathlib import Path\nimport pandas as pd\n\n# Ensure project root is in sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.build_manifest import DATASET_SPECS, resolve_dataset_root\n\n\nclass TestDatasetManifests(unittest.TestCase):\n    """Rigorous validation tests for dataset manifests and leak-proof splits."""\n\n    @classmethod\n    def setUpClass(cls):\n        cls.manifest_dir = PROJECT_ROOT / \'manifests\'\n        cls.seg_manifest_path = cls.manifest_dir / \'segmentation_manifest.csv\'\n        cls.rdd_manifest_path = cls.manifest_dir / \'rdd2022es_manifest.csv\'\n\n        if not cls.seg_manifest_path.exists():\n            raise FileNotFoundError(f"Missing segmentation manifest at {cls.seg_manifest_path}. Run src/build_manifest.py first.")\n\n        cls.df_seg = pd.read_csv(cls.seg_manifest_path)\n        cls.df_rdd = pd.read_csv(cls.rdd_manifest_path) if cls.rdd_manifest_path.exists() else None\n\n    def test_01_no_group_id_leakage_within_sources(self):\n        """(a) Assert that no group_id appears in more than one split within any source."""\n        for src_name, df_src in self.df_seg.groupby(\'source\'):\n            splits = df_src[\'split\'].unique()\n            group_split_counts = df_src.groupby(\'group_id\')[\'split\'].nunique()\n            leaked_groups = group_split_counts[group_split_counts > 1]\n\n            self.assertEqual(\n                len(leaked_groups), 0,\n                f"Data leakage detected in source \'{src_name}\'! Leaked group_ids: {list(leaked_groups.index[:5])}"\n            )\n            print(f"[PASS] Zero group_id leakage in \'{src_name}\' across {len(splits)} splits.")\n        if self.df_rdd is not None:\n            rdd_leak = self.df_rdd.groupby(\'group_id\')[\'split\'].nunique()\n            leaked_rdd = rdd_leak[rdd_leak > 1]\n            self.assertEqual(len(leaked_rdd), 0, f"Leakage detected in RDD2022ES! Leaked groups: {list(leaked_rdd.index[:5])}")\n            print(f"[PASS] Zero group_id leakage in \'rdd2022es\' across 3 splits.")\n\n    def test_02_every_manifest_path_exists_on_disk(self):\n        """(b) Assert that all image and mask/label file paths recorded exist on disk."""\n        # Check segmentation manifest\n        for src_name, df_src in self.df_seg.groupby(\'source\'):\n            spec = DATASET_SPECS.get(src_name)\n            if not spec:\n                continue\n            root = resolve_dataset_root(spec[\'owner\'], spec[\'name\'])\n            if not root or not root.exists():\n                print(f"Skipping disk path check for \'{src_name}\' (dataset root not mounted in current environment).")\n                continue\n\n            for idx, row in df_src.iterrows():\n                img_file = root / row[\'image_path\']\n                self.assertTrue(img_file.exists(), f"Missing image file on disk: {img_file}")\n                \n                if pd.notna(row[\'mask_path_or_label_path\']) and row[\'mask_path_or_label_path\'] != "":\n                    mask_file = root / row[\'mask_path_or_label_path\']\n                    self.assertTrue(mask_file.exists(), f"Missing mask/label file on disk: {mask_file}")\n\n            print(f"[PASS] All {len(df_src)} image and mask paths verified on disk for \'{src_name}\'.")\n\n        # Check RDD manifest if dataset mounted\n        if self.df_rdd is not None:\n            spec_rdd = DATASET_SPECS[\'rdd2022es\']\n            root_rdd = resolve_dataset_root(spec_rdd[\'owner\'], spec_rdd[\'name\'])\n            if root_rdd and root_rdd.exists():\n                sample_rdd = self.df_rdd.sample(min(100, len(self.df_rdd)), random_state=42)\n                for idx, row in sample_rdd.iterrows():\n                    img_file = root_rdd / row[\'image_path\']\n                    self.assertTrue(img_file.exists(), f"Missing image file in RDD2022ES: {img_file}")\n                print(f"[PASS] Sampled paths verified on disk for \'rdd2022es\'.")\n\n    def test_03_split_sizes_match_specifications(self):\n        """(c) Assert split sizes match the established targets."""\n        # 1. Pothole: 780 images, test=60, train+val=720 (240 groups)\n        df_p = self.df_seg[self.df_seg[\'source\'] == \'pothole\']\n        if len(df_p) > 0:\n            self.assertEqual(len(df_p), 780, f"Expected 780 pothole images, found {len(df_p)}")\n            self.assertEqual(len(df_p[df_p[\'split\'] == \'test\']), 60, "Expected 60 test images for pothole")\n            self.assertEqual(len(df_p[df_p[\'split\'] == \'val\']) + len(df_p[df_p[\'split\'] == \'train\']), 720)\n            print(f"[PASS] Pothole split sizes verified: {dict(df_p[\'split\'].value_counts())}")\n\n        # 2. Road Crack: 435 images, 22 scenes\n        df_rc = self.df_seg[self.df_seg[\'source\'] == \'road_crack\']\n        if len(df_rc) > 0:\n            self.assertEqual(len(df_rc), 435, f"Expected 435 road-crack images, found {len(df_rc)}")\n            self.assertEqual(df_rc[\'group_id\'].nunique(), 22, "Expected 22 unique scenes in road-crack")\n            print(f"[PASS] Road-crack split sizes verified: {dict(df_rc[\'split\'].value_counts())}")\n\n        # 3. CRACK500: 471 images (329 train, 70 val, 72 test)\n        df_c = self.df_seg[self.df_seg[\'source\'] == \'crack500\']\n        if len(df_c) > 0:\n            self.assertEqual(len(df_c), 471, f"Expected 471 crack500 images, found {len(df_c)}")\n            self.assertEqual(len(df_c[df_c[\'split\'] == \'train\']), 329)\n            self.assertEqual(len(df_c[df_c[\'split\'] == \'val\']), 70)\n            self.assertEqual(len(df_c[df_c[\'split\'] == \'test\']), 72)\n            print(f"[PASS] CRACK500 split sizes verified: {dict(df_c[\'split\'].value_counts())}")\n\n        # 4. RDD2022ES: 25,600 images (80/10/10)\n        if self.df_rdd is not None:\n            self.assertEqual(len(self.df_rdd), 25600, f"Expected 25600 RDD images, found {len(self.df_rdd)}")\n            self.assertEqual(len(self.df_rdd[self.df_rdd[\'split\'] == \'train\']), 20480)\n            self.assertEqual(len(self.df_rdd[self.df_rdd[\'split\'] == \'val\']), 2560)\n            self.assertEqual(len(self.df_rdd[self.df_rdd[\'split\'] == \'test\']), 2560)\n            print(f"[PASS] RDD2022ES split sizes verified: {dict(self.df_rdd[\'split\'].value_counts())}")\n\n    def test_04_rdd_mirrored_pairs_share_split(self):\n        """(d) Assert that mirrored RDD2022ES pairs strictly share the exact same split."""\n        if self.df_rdd is None:\n            print("Skipping RDD mirror test (RDD manifest not loaded).")\n            return\n\n        split_per_group = self.df_rdd.groupby(\'group_id\')[\'split\'].nunique()\n        self.assertTrue((split_per_group == 1).all(), "Some mirrored RDD pairs were assigned to different splits!")\n\n        # Verify exact pair count\n        images_per_group = self.df_rdd.groupby(\'group_id\')[\'image_path\'].count()\n        self.assertTrue((images_per_group == 2).all(), "Every RDD group_id must contain exactly 2 images (original and mirrored).")\n        print(f"[PASS] All 12,800 mirrored RDD pairs strictly share the exact same split.")\n\n\ndef run_tests():\n    suite = unittest.TestLoader().loadTestsFromTestCase(TestDatasetManifests)\n    runner = unittest.TextTestRunner(verbosity=2)\n    result = runner.run(suite)\n    return result.wasSuccessful()\n\n\nif __name__ == \'__main__\':\n    success = run_tests()\n    sys.exit(0 if success else 1)\n', encoding='utf-8')
    print('[OK] Written tests/test_manifest.py')

print('Environment initialized. Project root:', PROJECT_ROOT)


In [ ]:
# ==============================================================================
# 2. GENERATE MANIFESTS
# ==============================================================================
from src.build_manifest import generate_all_manifests, DATASET_SPECS, resolve_dataset_root

manifests = generate_all_manifests(output_dir=Path('manifests'))
print('\nManifest generation completed.')


In [ ]:
# ==============================================================================
# 3. RUN MANIFEST TEST SUITE
# ==============================================================================
import unittest
import importlib.util

print('=' * 80)
print('RUNNING UNIT TESTS FOR MANIFESTS & LEAK-PROOF SPLITS')
print('=' * 80)

# Load test module reliably
test_spec = importlib.util.spec_from_file_location('test_manifest', 'tests/test_manifest.py')
test_mod = importlib.util.module_from_spec(test_spec)
test_spec.loader.exec_module(test_mod)
TestDatasetManifests = test_mod.TestDatasetManifests

suite = unittest.TestLoader().loadTestsFromTestCase(TestDatasetManifests)
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)

if not test_result.wasSuccessful():
    raise RuntimeError('One or more manifest tests failed!')
else:
    print('\nAll manifest validation tests passed successfully!')


In [ ]:
# ==============================================================================
# 4. FOREGROUND PIXEL FRACTION & SPLIT STATISTICS
# ==============================================================================
import numpy as np
import pandas as pd
from PIL import Image
from src.data_rules import load_standardized_mask

seg_df = pd.read_csv('manifests/segmentation_manifest.csv')

stats_rows = []
sources = ['road_crack', 'crack500', 'pothole']
splits = ['train', 'val', 'test']

for src_name in sources:
    spec = DATASET_SPECS[src_name]
    root = resolve_dataset_root(spec['owner'], spec['name'])
    if root is None:
        print(f'Warning: Dataset root for {src_name} not found.')
        continue
        
    for sp in splits:
        sub = seg_df[(seg_df['source'] == src_name) & (seg_df['split'] == sp)]
        n_images = len(sub)
        n_groups = sub['group_id'].nunique()
        
        total_pixels = 0
        fg_pixels = 0
        
        for _, row in sub.iterrows():
            w, h = int(row['width']), int(row['height'])
            total_pixels += (w * h)
            
            lbl_rel = row['mask_path_or_label_path']
            if pd.notna(lbl_rel) and lbl_rel != '':
                lbl_p = root / lbl_rel
                if lbl_p.exists():
                    try:
                        mask = load_standardized_mask(src_name, lbl_p, img_shape=(h, w))
                        fg_pixels += int(np.sum(mask == 1))
                    except Exception:
                        pass
                        
        fg_fraction_pct = (fg_pixels / total_pixels * 100.0) if total_pixels > 0 else 0.0
        defect_type = 'Crack' if sub['labels_crack'].iloc[0] else 'Pothole'
        
        stats_rows.append({
            'Source': src_name,
            'Split': sp,
            'Images': n_images,
            'Unique Groups': n_groups,
            'Defect Channel': defect_type,
            'Total Pixels': f'{total_pixels:,}',
            'Foreground Pixels': f'{fg_pixels:,}',
            'Foreground Pixel Fraction (%)': f'{fg_fraction_pct:.3f}%'
        })

df_stats = pd.DataFrame(stats_rows)
print('=' * 100)
print('SEGMENTATION SOURCES: SPLIT METRICS AND FOREGROUND PIXEL FRACTIONS')
print('=' * 100)
display(df_stats)


In [ ]:
# ==============================================================================
# 5. VISUAL INSPECTION: 4 SAMPLES PER SEGMENTATION SOURCE
#    Image | Converted Mask (Crack = Red, Pothole = Blue)
#    Non-overlapping clean titles and layouts
# ==============================================================================
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)

COLOR_CRACK_RGB = np.array([255, 30, 30], dtype=np.uint8)    # Red
COLOR_POTHOLE_RGB = np.array([30, 120, 255], dtype=np.uint8)  # Blue

def colorize_binary_mask(mask: np.ndarray, color_rgb: np.ndarray) -> np.ndarray:
    h, w = mask.shape[:2]
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    rgb[mask == 1] = color_rgb
    return rgb

for src_name in ['road_crack', 'crack500', 'pothole']:
    spec = DATASET_SPECS[src_name]
    root = resolve_dataset_root(spec['owner'], spec['name'])
    if root is None:
        continue
        
    sub = seg_df[seg_df['source'] == src_name]
    sub_with_mask = sub[sub['mask_path_or_label_path'].notna() & (sub['mask_path_or_label_path'] != '')]
    sample_df = sub_with_mask.sample(min(4, len(sub_with_mask)), random_state=42)
    
    is_crack = sub['labels_crack'].iloc[0]
    color = COLOR_CRACK_RGB if is_crack else COLOR_POTHOLE_RGB
    defect_name = 'Crack' if is_crack else 'Pothole'
    color_name = 'Red' if is_crack else 'Blue'
    
    fig, axes = plt.subplots(4, 2, figsize=(11, 14))
    fig.suptitle(f'Dataset: {src_name} — 4 Visual Verification Samples\nDefect: {defect_name} (Color: {color_name})',
                 fontsize=14, weight='bold', y=0.995)
    
    for idx, (_, row) in enumerate(sample_df.iterrows()):
        img_p = root / row['image_path']
        lbl_p = root / row['mask_path_or_label_path']
        
        img = Image.open(img_p).convert('RGB')
        w, h = img.size
        mask = load_standardized_mask(src_name, lbl_p, img_shape=(h, w))
        colored_mask = colorize_binary_mask(mask, color)
        fg_pct = (np.sum(mask == 1) / mask.size) * 100.0
        
        stem = Path(row['image_path']).stem
        if len(stem) > 28:
            stem = stem[:25] + '...'
            
        # Left: Original Image
        ax_img = axes[idx, 0]
        ax_img.imshow(img)
        ax_img.set_title(f"[{row['split'].upper()}] Image: {stem}\n(Group: {row['group_id']})",
                         fontsize=9.5, pad=6)
        ax_img.axis('off')
        
        # Right: Converted Mask
        ax_mask = axes[idx, 1]
        ax_mask.imshow(colored_mask)
        ax_mask.set_title(f"[{row['split'].upper()}] Converted {defect_name} ({color_name})\nFG Area: {fg_pct:.2f}% | Size: {w}x{h}",
                          fontsize=9.5, pad=6)
        ax_mask.axis('off')
        
    plt.subplots_adjust(top=0.94, hspace=0.35, wspace=0.15)
    plt.show()
